# Test 1 of the Occlusion Analysis in AliceNet (Unencrypted Version)

This Jupyter Notebook consists of the tests for the three algorithms for the unencrypted Occlusion Analysis on AliceNet.

AliceNet is a neural network that was trained on the MNIST dataset and is provided by the CrypTen tutorials.

The three algorithms used are the sliding-window algorithm, the hierarchical algorithm, and the Monte Carlo algorithm. They use two parties for the inference, but none of them encrypt their data.

The cells of the presets must be run before the algorithms of the occlusion analysis. The tests of the algorithms don't need to be executed in a specific order.

Please note that this code was used to test the run time of the code and the communication needed between the parties. The description of the algorithms can be seen in the non-test Jupyter notebooks.

## Presets

The presets contain all the important imports, the download of the MNIST data, the network from Alice and a function for computing the accuracy.

The download of the MNIST data was provided by the CrypTen tutorials for this use case. It gives the training data to party one and the test data to party two.

The class AliceNet defines the network Alice owns. It is also provided by the CrypTen tutorials. The init method defines the neural network, while the forward method shows the forward pass of the network.

The function compute_accuracy is also provided by the CrypTen tutorials. The parameters are the output and the labels, whereas the output is the classification of the neural network and the label is the correct class indices. It computes and returns the accuracy of the classification from a model. 

In [ ]:
# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Standard Libraries
import time
import os

# Third-Party Libraries
import numpy as np
import matplotlib.pyplot as plt
import psutil

# CrypTen
import crypten
import crypten.mpc as mpc
import crypten.communicator as comm
from crypten.config import cfg

# Initializing CrypTen
crypten.init()
torch.set_num_threads(1)

In [ ]:
# Run script that downloads the publicly available MNIST data, and splits the data as required. (from CrypTen tutorials)
%run ./mnist_utils.py --option train_v_test

In [ ]:
# Define Alice's network (from CrypTen tutorials)
class AliceNet(nn.Module):
    def __init__(self):
        super(AliceNet, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 10)
 
    def forward(self, x):
        out = self.fc1(x)
        out = F.relu(out)
        out = self.fc2(out)
        out = F.relu(out)
        out = self.fc3(out)
        return out

In [ ]:
# From CrypTen tutorials
def compute_accuracy(output, labels):
    pred = output.argmax(1)
    correct = pred.eq(labels)
    correct_count = correct.sum(0, keepdim=True).float()
    accuracy = correct_count.mul_(100.0 / output.size(0))
    return accuracy

## The sliding window algorithm

This function implements the unencrypted version of the sliding-window algorithm. 

The following parameters can be specified: n is the number of repetitions the code should run. The patch_size can be changed to set the size of the patch. The occluded value can also be adjusted. 

The time is taken in three different phases and the model-loading. The model loading starts before the model is loaded and ends right after it. In order to measure the time needed for the preparation, the time starts before the data is loaded and ends right after the inference with the original picture is done. Right after the loop for the occlusion analysis, the starting point for measuring the time starts. The time stops after the calculated difference of the outputs is put in the heatmap. The value needs to be stored per iteration. Therefore, the values get appended to an array that holds the numbers for each iteration. After the occlusion analysis, the measurement of the visualization starts. The starting point is here before the heatmap gets normalized, and the endpoint is after the heatmap is shown.

After every repetition, the values of the current repetition will be added into an array that contains the values for all the times of the different phases. After the last repetition, these values will be used to calculate the mean, standard deviation, the maximum, and the minimum of each phase. The method calculate_stats is used for that. Please note that the values of the occlusion analysis are in a multidimensional array; therefore, the data needs to be flattened first. The method takes care of that. 

After the calculation is done, the results will be printed. Additionally, three graphs that show the occlusion analysis better are shown. They show the process of each iteration in relation to the time, the cumulative process in relation to time, and the average time needed for each iteration.

In [ ]:
from statistics import mean, stdev, mean

# Worldsize is 2 because of the two parties
@mpc.run_multiprocess(world_size=2)
def occlusion_analysis():
    """
    Performs the standard case of occlusion analysis on AliceNet.
    The algorithm is a version of the sliding-window algorithm.
    The result is a heatmap.
    """
    n = 1000  # Number of repetitions, can be changed

    # Arrays to store the calculated metrics for the different phases
    all_load_times = []
    
    all_prep_times = []
    
    all_occl_times = []  
    
    all_vis_times = []
 
    all_iteration_counts = []

    # Loop for the number of times the code should be tested
    for run in range(n):
        crypten.print(f"\nCurrent run {run+1}/{n}")

        # Arrays for the Occlusion Analysis per iteration
        run_occl_times = []
    
        # Start time for loading
        load_time_start = time.time()*1000

        # Load model
        model = torch.load('models/tutorial4_alice_model.pth', weights_only=False)
        model.eval() # Put model in inference phase

        # End time for loading
        load_time_end = time.time()*1000

        # Calculate the time of the loading
        load_time = (load_time_end - load_time_start)
    
        # Start time for prep
        prep_time_start = time.time()*1000
        
        # Load data
        data = torch.load('/tmp/bob_test.pth')[0].unsqueeze(0) # Unsqueeze was needed to add another dimension
        image = data.view(28, 28)
        
        # Parameters for Occlusion Analysis 
        # - patch size
        # - occluded_value
        # - heatmap
        patch_size = 4 # Can be changed
        occluded_value = 0 # Can be changed
        heatmap = torch.zeros((image.shape[0], image.shape[1]))
    
        # Computation of original output of the original picture
        with torch.no_grad(): # Allows quicker computation
            original_output = model(image.view(1, 784)) # View is needed for the input of the model

        # End time for prep
        prep_time_end = time.time()*1000
    
        # Calculate the time of the prep
        prep_time = (prep_time_end - prep_time_start)
    
        num_it = 1 # Counter for number of iterations
    
        # Iterating over the picture
        for y in range(0, image.shape[0], patch_size):
            for x in range(0, image.shape[1], patch_size):
    
                # Start time for Occlusion Analysis
                occl_time_start = time.time()*1000
                
                # Edge treatment:
                # The normal patch size is used 
                # Unless the patch doesn't entirely fit in the edge
                # Then the patch_size is the image.size - the current pixel position
                better_patch_size_y = min(patch_size, image.shape[0] - y)
                better_patch_size_x = min(patch_size, image.shape[1] - x)
                
                # Used if patch would be too small
                if better_patch_size_y < 1 or better_patch_size_x < 1:
                    continue
                    
                # Clone the picture and put the mask on the position
                occluded_image = image.clone()
                occluded_image[y : y + better_patch_size_y, x : x + better_patch_size_x] = occluded_value
    
                # Calculating the occluded_output
                with torch.no_grad():
                    occluded_output = model(occluded_image.view(1, 784))
                
                # Calculating the absolute difference of the original output and the occluded output
                diff = (original_output - occluded_output).abs().sum().item()
    
                # Put the difference in the heatmap
                heatmap[y : y + better_patch_size_y, x : x + better_patch_size_x] += diff

                # End time for Occlusion Analysis
                occl_time_end = time.time()*1000
    
                # Calculate the time of the Occlusion Analysis
                occl_time = occl_time_end - occl_time_start
    
                # Put in the calculated metrics for the current iteration
                run_occl_times.append(occl_time)
    
                crypten.print(f"\rNumber of iterations: {num_it}", end='', flush=True)
                num_it += 1

        # Store iteration count for this run
        all_iteration_counts.append(num_it - 1)  
    
        # Start time for for visualization
        vis_time_start = time.time()*1000
        
        # Normalizing the heatmap
        heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
    
        # Visualizing the heatmap
        plt.imshow(heatmap_norm, cmap='hot')
        plt.title("Occlusion Analysis Heatmap from Sliding-Window Algorithm (Unencrypted Version)")
        plt.colorbar()
        plt.show()

        # End time for visualization
        vis_time_end = time.time()*1000
    
        # Calculate the time of the visualization
        vis_time = vis_time_end - vis_time_start

        # Add the values for the run
        all_load_times.append(load_time)
    
        all_prep_times.append(prep_time)
        
        all_occl_times.append(run_occl_times)
 
        all_vis_times.append(vis_time)

        # Only after the last run the results should be presented and calculated
        if run + 1 == n:
            # Calculate mean, std, min and max for given data
            def calculate_stats(data):

                # Checking if the array has more than one dimension
                if isinstance(data[0], list): 
                    flat_data = [item for sublist in data for item in sublist]
                    return {
                        'mean': mean(flat_data),
                        'stdev': stdev(flat_data) if len(flat_data) > 1 else 0,
                        'min': min(flat_data),
                        'max': max(flat_data),
                        'total': sum(flat_data)
                    }
                else:
                    return {
                        'mean': mean(data),
                        'stdev': stdev(data) if len(data) > 1 else 0,
                        'min': min(data),
                        'max': max(data),
                        'total': sum(data)
                    }
            
            # Calculate the values for every metric
            load_time_stats = calculate_stats(all_load_times)
  
            prep_time_stats = calculate_stats(all_prep_times)
            
            occl_time_stats = calculate_stats([item for sublist in all_occl_times for item in sublist])
           
            vis_time_stats = calculate_stats(all_vis_times)
           
            # Show the calculated statistics
            crypten.print("\n\n==Values of the test==")
            crypten.print(f"\nIterations per run: {sum(all_iteration_counts)/n:.1f}")
            crypten.print(f"\nTimes the code was tested: {n}")

            crypten.print("\nLoading:")
            crypten.print(f"Time in ms: Mean={load_time_stats['mean']:.3f}, Stdev={load_time_stats['stdev']:.3f}, Min={load_time_stats['min']:.3f}, Max={load_time_stats['max']:.3f}")
            
            crypten.print("\nPrep:")
            crypten.print(f"Time in ms: Mean={prep_time_stats['mean']:.3f}, Stdev={prep_time_stats['stdev']:.3f}, Min={prep_time_stats['min']:.3f}, Max={prep_time_stats['max']:.3f}")
            
            crypten.print("\nActual Occlusion Analysis:")
            crypten.print(f"Time in ms: Mean={occl_time_stats['mean']:.3f}, Stdev={occl_time_stats['stdev']:.3f}, Min={occl_time_stats['min']:.3f}, Max={occl_time_stats['max']:.3f}, Total={occl_time_stats['total']}")
            
            crypten.print("\nVisualization:")
            crypten.print(f"Time in ms: Mean={vis_time_stats['mean']:.3f}, Stdev={vis_time_stats['stdev']:.3f}, Min={vis_time_stats['min']:.3f}, Max={vis_time_stats['max']:.3f}")
            
            # Plot iteration times
            plt.figure(figsize=(10, 6))
            for i, run_times in enumerate(all_occl_times):
                plt.plot(run_times, label=f'Run {i+1}')
            plt.xlabel('Number of Iterations')
            plt.ylabel('Time in ms')
            plt.title('Needed time for Iteration in Occlusion Analysis')
            plt.legend()
            plt.show()
            
            # Plot cum. iteration times
            plt.figure(figsize=(10, 6))
            for i, run_times in enumerate(all_occl_times):
                cumulative_times = np.cumsum(run_times)
                plt.plot(cumulative_times, label=f'Run {i+1}')
            plt.xlabel('Number of Iterations')
            plt.ylabel('Cumulative Time in ms')
            plt.title('Cumulative time for Occlusion Analysis')
            plt.legend()
            plt.grid(True)
            plt.show()  

            # Plots mean of the iterations over all runs
            plt.figure(figsize=(10, 6))
            avg_times = np.mean(np.array(all_occl_times), axis=0)
            plt.plot(avg_times, label = f'Average time per Iteration')
            plt.xlabel('Number of Iterations')
            plt.ylabel('Time in ms')
            plt.title('Average time per Iteration')
            plt.legend()
            plt.show()

occlusion_analysis()

## The hierarchical occlusion algorithm

This function implements the unencrypted version of the hierarchical occlusion algorithm. 

The following parameters can be specified: n is the number of repetitions the code should run. The initial_patch and the minimal_patch can be changed to set the size of these patches. The occluded_value and the threshold can also be adjusted. 

The time is taken in three different phases and the model-loading. The model loading starts before the model is loaded and ends right after it. In order to measure the time needed for the preparation, the time starts before the data is loaded and ends right after the inference with the original picture is done. Right after the loop for the occlusion analysis, the starting point for measuring the time starts. The time stops after the decision of if the patch should be divided further is calculated. The value needs to be stored per iteration. Therefore, the values get appended to an array that holds the numbers for each iteration. After the occlusion analysis, the measurement of the visualization starts. The starting point is here before the heatmap gets normalized, and the endpoint is after the heatmap is shown.

After every repetition, the values of the current repetition will be added into an array that contains the values for all the times of the different phases. After the last repetition, these values will be used to calculate the mean, standard deviation, the maximum, and the minimum of each phase. The method calculate_stats is used for that. Please note that the values of the occlusion analysis are in a multidimensional array; therefore, the data needs to be flattened first. The method takes care of that. 

After the calculation is done, the results will be printed. Additionally, three graphs that show the occlusion analysis better are shown. They show the process of each iteration in relation to the time, the cumulative process in relation to time, and the average time needed for each iteration.

In [ ]:
from statistics import mean, stdev

# Worldsize is 2 because of the two parties
@mpc.run_multiprocess(world_size=2)
def hierarchical_occlusion():
    """
    Performs a hierarchical occlusion analysis on AliceNet.
    The algorithm identifies regions that are important for the model and partitions the patch.
    The result is a heatmap.
    """
    n = 1000  # Number of repetitions, can be changed

    # Arrays to store the calculated metrics for the different phases
    all_load_times = []
    
    all_prep_times = []
 
    all_occl_times = []  

    all_vis_times = []
   
    all_iteration_counts = []

    # Loop for the number of times the code should be tested
    for run in range(n):
        crypten.print(f"\nCurrent run {run+1}/{n}")

        # Arrays for the Occlusion Analysis per iteration
        run_occl_times = []
        
        # Start time for loading
        load_time_start = time.time()*1000

        # Load model
        model = torch.load('models/tutorial4_alice_model.pth', weights_only=False)
        model.eval() # Put model in inference phase

        # End time for loading
        load_time_end = time.time()*1000

        # Calculate the time of the loading
        load_time = (load_time_end - load_time_start)

        # Start time for prep
        prep_time_start = time.time()*1000
    
        # Load data
        data = torch.load('/tmp/bob_test.pth')[0].unsqueeze(0) # Unsqueeze was needed to add another dimension
        image = data.view(28, 28)
        
        # Parameters for Occlusion Analysis 
        # - initial_patch
        # - minimal_patch
        # - occluded_value
        # - threshold
        # - heatmap
        initial_patch = 8 # Can be changed
        minimal_patch = 4 # Can be changed
        occluded_value = 0 # Can be changed
        threshold = 0.001 # Can be changed
        heatmap = torch.zeros((image.shape[0], image.shape[1]))
    
        # Computation of original output of the original picture
        with torch.no_grad(): # Allows quicker computation
            original_output = model(image.view(1, 784)) # View is needed for the input of the model

        # End time for prep
        prep_time_end = time.time()*1000
    
        # Calculate the time of the prep
        prep_time = (prep_time_end - prep_time_start)
    
        num_it = 1 # Counter for number of iterations
        
        def patch_partition(starting_point, current_patch_size, heatmap):
            """
            Partitions the patch if the difference of the original output and occluded output is important.
            Args:
                starting_point: top-left corner of the current patch
                current_patch_size: Size of the current patch
                heatmap: Contains the difference of the original output and occluded output
            Returns:
                heatmap
            """  

            # Start time for Occlusion Analysis
            occl_time_start = time.time()*1000
            
            nonlocal num_it
            y, x = starting_point
    
            # Edge treatment:
            # The normal patch size is used 
            # Unless the patch doesn't entirely fit in the edge
            # Then the patch_size is the image.size - the current pixel position
            better_patch_size_y = min(current_patch_size, image.shape[0] - y)
            better_patch_size_x = min(current_patch_size, image.shape[1] - x)
    
            # Used if patch would be too small
            if better_patch_size_y < 1 or better_patch_size_x < 1:
                return heatmap      
    
            # Clone the picture and put the mask on the position
            occluded_image = image.clone()
            occluded_image[y : y + better_patch_size_y, x : x + better_patch_size_x] = occluded_value
    
            # Calculating the occluded_output
            with torch.no_grad():
                occluded_output = model(occluded_image.view(1, 784))
                
            # Calculating the absolute difference of the original output and the occluded output
            diff = (original_output - occluded_output).abs().sum().item()
            
            # Put the difference in the heatmap
            heatmap[y : y + better_patch_size_y, x : x + better_patch_size_x] += diff
    
            # Calculates the difference but in the range 0 to 1
            comp = ((original_output.softmax(1) - occluded_output.softmax(1)).abs().sum())/ 2
    
            # Decision if the patch should be divided
            # Difference needs to be bigger than threshold 
            # and patch must be bigger than smallest patch
            differentiate = (comp > threshold) and (float(current_patch_size)/2.0 >= float(minimal_patch))

            # End time for Occlusion Analysis
            occl_time_end = time.time()*1000
    
            # Calculate the time of the Occlusion Analysis
            occl_time = occl_time_end - occl_time_start

            # Put in the calculated metrics for the current iteration
            run_occl_times.append(occl_time)
             
            # If decision is true, divide the patch 
            # Set the position for the patches
            if differentiate:
                half_patch = current_patch_size // 2
                # Calculate possible positions and set new positions
                for poss_y in [0, half_patch]:
                    for poss_x in [0, half_patch]:
                        new_y = poss_y + y
                        new_x = poss_x + x
                        # Check if patch position is bigger than picture
                        if (new_y < image.shape[0] and new_x < image.shape[1]):
                            heatmap = patch_partition((new_y, new_x), half_patch, heatmap)
            
            crypten.print(f"\rNumber of iterations: {num_it}", end='', flush=True)
            num_it += 1
            
            return heatmap
        
        # Start of Occlusion Analysis
        for y in range(0, image.shape[0], initial_patch):
            for x in range(0, image.shape[1], initial_patch):
                heatmap = patch_partition((y,x), initial_patch, heatmap)     
                
        # Store iteration count for this run
        all_iteration_counts.append(num_it - 1)  
        
        # Start time for for visualization
        vis_time_start = time.time()*1000

        # Normalizing the heatmap
        heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
        
        # Visualising the heatmap
        plt.imshow(heatmap_norm, cmap='hot')
        plt.title("Occlusion Analysis Heatmap from Hierarchical Occlusion Algorithm (Unencrypted Model)")
        plt.colorbar()
        plt.show()

        # End time for visualization
        vis_time_end = time.time()*1000
    
        # Calculate the time of the visualization
        vis_time = vis_time_end - vis_time_start
    
        # Add the values for the run
        all_load_times.append(load_time)
    
        all_prep_times.append(prep_time)
        
        all_occl_times.append(run_occl_times)
        
        all_vis_times.append(vis_time)

        # Only after the last run the results should be presented and calculated
        if run + 1 == n:
            # Calculate mean, std, min and max for given data
            def calculate_stats(data):

                # Checking if the array has more than one dimension
                if isinstance(data[0], list): 
                    flat_data = [item for sublist in data for item in sublist]
                    return {
                        'mean': mean(flat_data),
                        'stdev': stdev(flat_data) if len(flat_data) > 1 else 0,
                        'min': min(flat_data),
                        'max': max(flat_data),
                        'total': sum(flat_data)
                    }
                else:
                    return {
                        'mean': mean(data),
                        'stdev': stdev(data) if len(data) > 1 else 0,
                        'min': min(data),
                        'max': max(data),
                        'total': sum(data)
                    }
            
            # Calculate the values for every metric
            load_time_stats = calculate_stats(all_load_times)
  
            prep_time_stats = calculate_stats(all_prep_times)

            occl_time_stats = calculate_stats([item for sublist in all_occl_times for item in sublist])
       
            vis_time_stats = calculate_stats(all_vis_times)
      
            # Show the calculated statistics
            crypten.print("\n\n==Values of the test==")
            crypten.print(f"\nIterations per run: {sum(all_iteration_counts)/n:.1f}")
            crypten.print(f"\nTimes the code was tested: {n}")

            crypten.print("\nLoading:")
            crypten.print(f"Time in ms: Mean={load_time_stats['mean']:.3f}, Stdev={load_time_stats['stdev']:.3f}, Min={load_time_stats['min']:.3f}, Max={load_time_stats['max']:.3f}")
            
            crypten.print("\nPrep:")
            crypten.print(f"Time in ms: Mean={prep_time_stats['mean']:.3f}, Stdev={prep_time_stats['stdev']:.3f}, Min={prep_time_stats['min']:.3f}, Max={prep_time_stats['max']:.3f}")
            
            crypten.print("\nActual Occlusion Analysis:")
            crypten.print(f"Time in ms: Mean={occl_time_stats['mean']:.3f}, Stdev={occl_time_stats['stdev']:.3f}, Min={occl_time_stats['min']:.3f}, Max={occl_time_stats['max']:.3f}, Total={occl_time_stats['total']}")
            
            crypten.print("\nVisualization:")
            crypten.print(f"Time in ms: Mean={vis_time_stats['mean']:.3f}, Stdev={vis_time_stats['stdev']:.3f}, Min={vis_time_stats['min']:.3f}, Max={vis_time_stats['max']:.3f}")
            
            # Plot iteration times
            plt.figure(figsize=(10, 6))
            for i, run_times in enumerate(all_occl_times):
                plt.plot(run_times, label=f'Run {i+1}')
            plt.xlabel('Number of Iterations')
            plt.ylabel('Time in ms')
            plt.title('Needed time for Iteration in Occlusion Analysis')
            plt.legend()
            plt.show()
            
            # Plot cum. iteration times
            plt.figure(figsize=(10, 6))
            for i, run_times in enumerate(all_occl_times):
                cumulative_times = np.cumsum(run_times)
                plt.plot(cumulative_times, label=f'Run {i+1}')
            plt.xlabel('Number of Iterations')
            plt.ylabel('Cumulative Time in ms')
            plt.title('Cumulative time for Occlusion Analysis')
            plt.legend()
            plt.grid(True)
            plt.show()  

            # Plots mean of the iterations over all runs
            plt.figure(figsize=(10, 6))
            avg_times = np.mean(np.array(all_occl_times), axis=0)
            plt.plot(avg_times, label = f'Average time per Iteration')
            plt.xlabel('Number of Iterations')
            plt.ylabel('Time in ms')
            plt.title('Average time per Iteration')
            plt.legend()
            plt.show()
        
hierarchical_occlusion()

## The Monte Carlo Occlusion Algorithm

This function implements the unencrypted version of the Monte Carlo algorithm. 

The following parameters can be specified: n is the number of repetitions the code should run. The patch_size can be changed to set the size of the patch. The occluded_value  and the num_simulations can also be adjusted. 

The time is taken in three different phases and the model-loading. The model loading starts before the model is loaded and ends right after it. In order to measure the time needed for the preparation, the time starts before the data is loaded and ends right after the possible positions for the patch are calculated. Right after the loop for the occlusion analysis, the starting point for measuring the time starts. The time stops after the calculated difference of the outputs is put in the heatmap. The value needs to be stored per iteration. Therefore, the values get appended to an array that holds the numbers for each iteration. After the occlusion analysis, the measurement of the visualization starts. The starting point is here before the heatmap gets divided by the number of simulations, and the endpoint is after the heatmap is shown.

After every repetition, the values of the current repetition will be added into an array that contains the values for all the times of the different phases. After the last repetition, these values will be used to calculate the mean, standard deviation, the maximum, and the minimum of each phase. The method calculate_stats is used for that. Please note that the values of the occlusion analysis are in a multidimensional array; therefore, the data needs to be flattened first. The method takes care of that. 

After the calculation is done, the results will be printed. Additionally, three graphs that show the occlusion analysis better are shown. They show the process of each iteration in relation to the time, the cumulative process in relation to time, and the average time needed for each iteration.

In [ ]:
from statistics import mean, stdev

# Worldsize is 2 because of the two parties
@mpc.run_multiprocess(world_size=2)
def monte_carlo_occlusion_analysis():
    """
    Performs a Monte Carlo occlusion analysis on AliceNet.
    The algorithm sets the patch to random places on the picture and calculates the difference.
    The result is a statistical heatmap.
    """
    n = 1000  # Number of repetitions, can be changed

    # Arrays to store the calculated metrics for the different phases
    all_load_times = []
    
    all_prep_times = []
    
    all_occl_times = []  

    all_vis_times = []
   
    all_iteration_counts = []

    # Loop for the number of times the code should be tested
    for run in range(n):
        crypten.print(f"\nCurrent run {run+1}/{n}")

        # Arrays for the Occlusion Analysis per iteration
        run_occl_times = []
        
        # Start time for loading
        load_time_start = time.time()*1000

        # Load model
        model = torch.load('models/tutorial4_alice_model.pth', weights_only=False)
        model.eval() # Put model in inference phase

        # End time for loading
        load_time_end = time.time()*1000

        # Calculate the time of the loading
        load_time = (load_time_end - load_time_start)

        # Start time for prep
        prep_time_start = time.time()*1000
    
        # Load data
        data = torch.load('/tmp/bob_test.pth')[0].unsqueeze(0) # Unsqueeze was needed to add another dimension
        image = data.view(28, 28)
    
        # Parameters for Occlusion Analysis 
        # - patch_size
        # - occluded_value
        # - num_simulations
        # - encrypted heatmap
        patch_size = 4 # Can be changed
        occluded_value = 0 # Can be changed
        num_simulations = 50  # Can be changed
        heatmap = torch.zeros((image.shape[0], image.shape[1]))
    
        # Computation of original output of the original picture
        with torch.no_grad(): # Allows quicker computation
            original_output = model(image.view(1, 784)) # View is needed for the input of the model

        # Actual_sim represents the number of actual simulations
        # Tried_sim is the number of tried simulations (those who failed and those who didn't
        # Max_sim is the number of maximum tries to avoid endless loop
        actual_sim = 0
        tried_sim = 0
        max_sim = num_simulations * 2 # Can be changed
    
        # Calculate the possible positions for the patch
        # By dividing the possible positions by the patch_size
        num_patches_y = (image.shape[0] - patch_size) // patch_size + 1
        num_patches_x = (image.shape[1] - patch_size) // patch_size + 1

        # End time for prep
        prep_time_end = time.time()*1000
    
        # Calculate the time of the prep
        prep_time = (prep_time_end - prep_time_start)
    
        num_it = 1 # Counter for number of iterations
    
        while actual_sim < num_simulations and tried_sim < max_sim:
    
            # Start time for Occlusion Analysis
            occl_time_start = time.time()*1000
            
            # Choose a position for the patch
            random_patch_y = np.random.randint(0, num_patches_y + 1)
            random_patch_x = np.random.randint(0, num_patches_x + 1)
    
            # Calculate the positions with the patch_size
            y = random_patch_y * patch_size
            x = random_patch_x * patch_size
    
            # Edge treatment:
            # The normal patch size is used 
            # Unless the patch doesn't entirely fit in the edge
            # Then the patch_size is the image.size - the current pixel position
            better_patch_size_y = min(patch_size, image.shape[0] - y + 1)
            better_patch_size_x = min(patch_size, image.shape[1] - x + 1)
    
            # Used if patch would be too small
            if better_patch_size_y < 1 or better_patch_size_x < 1:
                    tried_sim += 1
                    continue
                
            # Clone the picture and put the mask on the position
            occluded_image = image.clone()
            occluded_image[y : y + better_patch_size_y, x : x + better_patch_size_x] = occluded_value
    
            # Calculating the occluded_output
            with torch.no_grad():
                occluded_output = model(occluded_image.view(1, 784))
                
            # Calculating the absolute difference of the original output and the occluded output
            diff = (original_output - occluded_output).abs().sum().item()
    
            # Put the difference in the heatmap
            heatmap[y : y + better_patch_size_y, x : x + better_patch_size_x] += diff

            # End time for Occlusion Analysis
            occl_time_end = time.time()*1000
    
            # Calculate the time of the Occlusion Analysis
            occl_time = occl_time_end - occl_time_start
    
            # Put in the calculated metrics for the current iteration
            run_occl_times.append(occl_time)
        
            crypten.print(f"\rNumber of iterations: {num_it}", end='', flush=True)
            num_it += 1
            tried_sim += 1
            actual_sim += 1
            
        # Store iteration count for this run     
        all_iteration_counts.append(num_it - 1) 
        
        # Start time for for visualization
        vis_time_start = time.time()*1000
        
        # Calculate the average influence
        heatmap /= actual_sim
    
        # Normalizing the heatmap
        heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
    
        # Visualise Heatmap 
        plt.imshow(heatmap_norm, cmap='hot')
        plt.title("Occlusion Analysis Heatmap from Monte-Carlo Algorithm (Unencrypted Model)")
        plt.colorbar()
        plt.show()

        # End time for visualization
        vis_time_end = time.time()*1000
    
        # Calculate the time of the visualization
        vis_time = vis_time_end - vis_time_start
    
        # Add the values for the run
        all_load_times.append(load_time)
        
        all_prep_times.append(prep_time)
         
        all_occl_times.append(run_occl_times)
        
        all_vis_times.append(vis_time)
    
        # Only after the last run the results should be presented and calculated
        if run + 1 == n:
            # Calculate mean, std, min and max for given data
            def calculate_stats(data):

                # Checking if the array has more than one dimension
                if isinstance(data[0], list): 
                    flat_data = [item for sublist in data for item in sublist]
                    return {
                        'mean': mean(flat_data),
                        'stdev': stdev(flat_data) if len(flat_data) > 1 else 0,
                        'min': min(flat_data),
                        'max': max(flat_data),
                        'total': sum(flat_data)
                    }
                else:
                    return {
                        'mean': mean(data),
                        'stdev': stdev(data) if len(data) > 1 else 0,
                        'min': min(data),
                        'max': max(data),
                        'total': sum(data)
                    }
            
            # Calculate the values for every metric
            load_time_stats = calculate_stats(all_load_times)
            
            prep_time_stats = calculate_stats(all_prep_times)
            
            occl_time_stats = calculate_stats([item for sublist in all_occl_times for item in sublist])
            
            vis_time_stats = calculate_stats(all_vis_times)
            
            # Show the calculated statistics
            crypten.print("\n\n==Values of the test==")
            crypten.print(f"\nIterations per run: {sum(all_iteration_counts)/n:.1f}")
            crypten.print(f"\nTimes the code was tested: {n}")

            crypten.print("\nLoading:")
            crypten.print(f"Time in ms: Mean={load_time_stats['mean']:.3f}, Stdev={load_time_stats['stdev']:.3f}, Min={load_time_stats['min']:.3f}, Max={load_time_stats['max']:.3f}")
            
            crypten.print("\nPrep:")
            crypten.print(f"Time in ms: Mean={prep_time_stats['mean']:.3f}, Stdev={prep_time_stats['stdev']:.3f}, Min={prep_time_stats['min']:.3f}, Max={prep_time_stats['max']:.3f}")
            
            crypten.print("\nActual Occlusion Analysis:")
            crypten.print(f"Time in ms: Mean={occl_time_stats['mean']:.3f}, Stdev={occl_time_stats['stdev']:.3f}, Min={occl_time_stats['min']:.3f}, Max={occl_time_stats['max']:.3f}, Total={occl_time_stats['total']}")
            
            crypten.print("\nVisualization:")
            crypten.print(f"Time in ms: Mean={vis_time_stats['mean']:.3f}, Stdev={vis_time_stats['stdev']:.3f}, Min={vis_time_stats['min']:.3f}, Max={vis_time_stats['max']:.3f}")
            
            # Plot iteration times
            plt.figure(figsize=(10, 6))
            for i, run_times in enumerate(all_occl_times):
                plt.plot(run_times, label=f'Run {i+1}')
            plt.xlabel('Number of Iterations')
            plt.ylabel('Time in ms')
            plt.title('Needed time for Iteration in Occlusion Analysis')
            plt.legend()
            plt.show()
            
            # Plot cum. iteration times
            plt.figure(figsize=(10, 6))
            for i, run_times in enumerate(all_occl_times):
                cumulative_times = np.cumsum(run_times)
                plt.plot(cumulative_times, label=f'Run {i+1}')
            plt.xlabel('Number of Iterations')
            plt.ylabel('Cumulative Time in ms')
            plt.title('Cumulative time for Occlusion Analysis')
            plt.legend()
            plt.grid(True)
            plt.show()  

            # Plots mean of the iterations over all runs
            plt.figure(figsize=(10, 6))
            avg_times = np.mean(np.array(all_occl_times), axis=0)
            plt.plot(avg_times, label = f'Average time per Iteration')
            plt.xlabel('Number of Iterations')
            plt.ylabel('Time in ms')
            plt.title('Average time per Iteration')
            plt.legend()
            plt.show()

monte_carlo_occlusion_analysis()

This code is used for cleaning the files up.

In [ ]:
import os

filenames = ['/tmp/alice_train.pth', 
             '/tmp/alice_train_labels.pth', 
             '/tmp/bob_test.pth', 
             '/tmp/bob_test_labels.pth']

for fn in filenames:
    if os.path.exists(fn): os.remove(fn)